# ECE1512 Project A: ABMIL

Parteek Dheri

Completed course notebook. The model, training and evaluation definitions are the definitions used in the reported experiments. This copy uses the frozen final-study splits, including patient-disjoint C17 development data. See `report/report.md` for the study and `reproduce.py` for the three-method ten-seed protocol. Running the notebook trains a new baseline, not an extension.


## Learning objectives

After this project you should be able to explain and implement:

1. `Dataset` vs. `DataLoader`, batching, shuffling.
2. `nn.Module`, trainable parameters, and `forward()`.
3. A classification loss and optimizer.
4. The training sequence `zero_grad -> forward -> loss -> backward -> step`.
5. The difference between `model.train()` and `model.eval()`.

## Setup

Install the dependencies in `requirements.txt` in a separate environment before running. Obtain the course feature files as described in `DATA.md`. This notebook does not install packages automatically.


In [ ]:
# Dependencies are installed explicitly, outside the notebook.
import sys


In [ ]:
import os, sys, copy, time, json
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Match the existing Jupyter container's two-CPU limit.
torch.set_num_threads(2)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False})

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Using CPU with', torch.get_num_threads(), 'PyTorch threads.')


### Your experiment controls

- **`DATASET`**: `C16` (2 classes), `C17` (4 classes), or `BRACS` (7 classes).
- **`RUN_MODE`**: `explore` for a short learning run, `baseline` for the full configured training.
- **`EXPLORE_EPOCHS`**: start at 2; try 5 after the first successful run.
- **`SHOW_WSI`**: draw image overlays if the original `.tif` files are available. The coordinate-only attention map works without them.

Restart the kernel and run all cells after changing the dataset or experiment settings. Environment variables below are optional overrides for automated tests.


In [ ]:
DATASET = os.environ.get('PROJECTA_DATASET', 'C16')
RUN_MODE = os.environ.get('PROJECTA_MODE', 'explore')
EXPLORE_EPOCHS = int(os.environ.get('PROJECTA_EPOCHS', '2'))
SHOW_WSI = True

if DATASET not in {'C16', 'C17', 'BRACS'}:
    raise ValueError('DATASET must be C16, C17, or BRACS')
if RUN_MODE not in {'explore', 'baseline'} or EXPLORE_EPOCHS < 1:
    raise ValueError('Choose explore/baseline and at least one exploration epoch.')

# Open this notebook from the repository root or a child directory.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next((p.resolve() for p in candidates
                    if (p / 'reference_splits/C17/splits.json').is_file()
                    and (p / 'dataloader.py').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Open this notebook from the Project A folder or course workspace.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

CONFIG_FILES = {
    'C16': 'camelyon16_medical_ssl_config.yml',
    'C17': 'camelyon17_medical_ssl_config.yml',
    'BRACS': 'bracs_medical_ssl_config.yml',
}
CONFIG_PATH = PROJECT_DIR / 'config' / CONFIG_FILES[DATASET]
print('Project:', PROJECT_DIR)
print('Config :', CONFIG_PATH)
print('Mode   :', RUN_MODE)


In [ ]:
from dataloader import build_HDF5_feat_dataset, mil_collate_fn
from utils import load_config, seed_everything, infer_feature_dim, classification_metrics
from wsi_visualization import visualize_wsi_attention

conf = load_config(str(CONFIG_PATH))
# YAML paths are relative to the project, not to the kernel's starting directory.
conf.data_dir = str((PROJECT_DIR / conf.data_dir).resolve())
conf.dataset_csv_dir = str((PROJECT_DIR / conf.dataset_csv_dir).resolve())
seed_everything(int(conf.seed))

if not Path(conf.data_dir).is_file():
    raise FileNotFoundError(
        f'Missing features: {conf.data_dir}\n'
        'See DATA.md for feature acquisition and placement.'
    )
NUM_EPOCHS = EXPLORE_EPOCHS if RUN_MODE == 'explore' else int(conf.train_epoch)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = PROJECT_DIR / 'results' / DATASET / f'{RUN_MODE}_{run_id}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
CLASS_NAMES = ['Normal', 'Tumor'] if DATASET == 'C16' else [
    f'Class {i}' for i in range(int(conf.n_class))
]
print(conf)
print(f'\n{RUN_MODE.upper()}: {NUM_EPOCHS} epochs, frozen final-study splits.')
if RUN_MODE == 'explore':
    print('Learning run only; do not report it as the completed 30-epoch baseline.')
print('Outputs:', RUN_DIR)

def show_and_save(fig, filename):
    fig.savefig(RUN_DIR / filename, bbox_inches='tight', dpi=140)
    plt.show()
    plt.close(fig)


# Step 1 — Dataset and DataLoader

A **Dataset** reads one slide's feature bag and label. A **DataLoader** chooses slides, shuffles training samples, and assembles batches.

Slides have different numbers of patches. `mil_collate_fn` pads bags and supplies a Boolean mask: `True` means a real patch. Our attention module will ignore padded patches.

The plots below inspect **labels and bag sizes only** across the predefined splits. Test predictions are not used to choose a model.


In [ ]:
from run_baselines import load_fixed_datasets
train_data, val_data, test_data = load_fixed_datasets(
    conf, PROJECT_DIR / 'reference_splits' / DATASET / 'splits.json')
conf.D_feat = infer_feature_dim(train_data)
splits = {'Train': train_data, 'Validation': val_data, 'Test': test_data}
print('Slides:', {name: len(dataset) for name, dataset in splits.items()})
example = train_data[0]
print('One bag:', tuple(example['input'].shape), '= [patches, features]')
print('Slide:', example['slide_id'], '| label:', example['label'])

# Read HDF5 metadata, not all feature arrays, for these dataset summaries.
split_labels, split_sizes = {}, {}
with h5py.File(conf.data_dir, 'r') as h5:
    for name, dataset in splits.items():
        split_labels[name] = [int(h5[s].attrs['label']) for s in dataset.data_names]
        split_sizes[name] = [int(h5[s]['feat'].shape[0]) for s in dataset.data_names]
label_counts = pd.DataFrame({
    name: Counter(labels) for name, labels in split_labels.items()
}).reindex(range(int(conf.n_class)), fill_value=0).fillna(0).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
label_counts.plot.bar(ax=axes[0], rot=0)
axes[0].set(title='Slide labels by split', xlabel='Class ID', ylabel='Slides')
axes[1].hist(list(split_sizes.values()), bins=20, label=list(splits))
axes[1].set(title='Slides contain different numbers of patches',
            xlabel='Patches per slide', ylabel='Slides')
axes[1].legend()
image = axes[2].imshow(example['input'][:32, :32].numpy(), aspect='auto', cmap='coolwarm')
axes[2].set(title='A small slice of one feature bag', xlabel='Feature dimension', ylabel='Patch')
fig.colorbar(image, ax=axes[2], label='Feature value')
show_and_save(fig, 'data_overview.png')

run_config = {
    **vars(conf), 'dataset_key': DATASET, 'run_mode': RUN_MODE,
    'epochs': NUM_EPOCHS, 'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'split_sizes': {name: len(dataset) for name, dataset in splits.items()},
}
(RUN_DIR / 'config.json').write_text(json.dumps(run_config, indent=2))
(RUN_DIR / 'splits.json').write_text(json.dumps({
    name: dataset.data_names for name, dataset in splits.items()
}, indent=2))


In [ ]:
batch_size = int(conf.B)
num_workers = int(conf.n_worker)
pin_memory = bool(conf.pin_memory) and torch.cuda.is_available()
loader_args = dict(
    batch_size=batch_size, num_workers=num_workers,
    pin_memory=pin_memory, collate_fn=mil_collate_fn,
)
print('Batch size:', batch_size, '| workers:', num_workers, '| pinned memory:', pin_memory)


In [ ]:
# Only training samples are shuffled. Evaluation visits every slide once.
train_loader = DataLoader(train_data, shuffle=True, **loader_args)
val_loader = DataLoader(val_data, shuffle=False, **loader_args)
test_loader = DataLoader(test_data, shuffle=False, **loader_args)


In [ ]:
batch = next(iter(train_loader))
print('input:', tuple(batch['input'].shape), batch['input'].dtype, '= [B, N, D]')
print('mask :', tuple(batch['mask'].shape), '= [B, N]')
print('label:', tuple(batch['label'].shape), batch['label'].dtype)
print('Real patches per slide:', batch['mask'].sum(dim=1).tolist())
print('Slide IDs:', batch['slide_id'])
print('\nB = slides in the batch; N = maximum patches; D = features per patch.')


**Questions for your report:**

- What is the difference between a `Dataset` and a `DataLoader`?
- Which Dataloader should use `shuffle=True`? Why?
- For a batch with input shape `[B, N, D]`, what do `B`, `N`, and `D` mean in this project?


# Step 2 - Implement the ABMIL model

For each patch feature $h_i$, the gated attention score follows Ilse et al. (2018):

$$
a_i = w^T\left[\tanh(Vh_i) \odot \sigma(Uh_i)\right].
$$

The attention logits are normalized over instances:

$$
\alpha_i = \frac{\exp(a_i)}{\sum_j \exp(a_j)},
$$

and the slide representation is

$$
z = \sum_i \alpha_i h_i.
$$

You will implement the learnable layers and the key `forward()` operations.

[Custom nn Modules](https://docs.pytorch.org/tutorials/beginner/examples_nn/polynomial_module.html)

[nn.Linear](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html)

[nn.modules.dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html)

[nn.ReLU](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html)

[torch.nn.functional.softmax (F.softmax)](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.softmax.html)

[torch.bmm](https://docs.pytorch.org/docs/2.14/generated/torch.bmm.html)

In [ ]:
class Classifier1FC(nn.Module):
    def __init__(self, n_channels, n_classes, droprate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(droprate) if droprate > 0 else nn.Identity()
        self.fc = nn.Linear(n_channels, n_classes)

    def forward(self, x):
        return self.fc(self.dropout(x))


class DimReduction(nn.Module):
    def __init__(self, n_channels, m_dim):
        super().__init__()
        self.fc = nn.Linear(n_channels, m_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.fc(x))


class AttentionGated(nn.Module):
    def __init__(self, L=256, D=128, K=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_weights_w = nn.Linear(D, K)

    def forward(self, x):
        # [B, N, L] -> [B, K, N]: one learned score per patch.
        a_v = self.attention_V(x)
        a_u = self.attention_U(x)
        return self.attention_weights_w(a_v * a_u).transpose(1, 2)


class ABMIL(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.dimreduction = DimReduction(conf.D_feat, conf.D_inner)
        self.attention = AttentionGated(conf.D_inner, conf.D_attn, K=1)
        self.classifier = Classifier1FC(conf.D_inner, conf.n_class, conf.droprate)

    def forward(self, x, return_attention=False, mask=None):
        # x: [B, N, D_feat]; mask: [B, N], True only for real patches.
        med_feat = self.dimreduction(x)       # [B, N, D_inner]
        a_logits = self.attention(med_feat)   # [B, 1, N]
        if mask is not None:
            if mask.shape != x.shape[:2] or mask.dtype != torch.bool:
                raise ValueError('mask must be Boolean with shape [B, N]')
            if not mask.any(dim=1).all():
                raise ValueError('Each slide must contain at least one real patch.')
            a_logits = a_logits.masked_fill(~mask.unsqueeze(1), float('-inf'))

        attention = F.softmax(a_logits, dim=-1)               # [B, 1, N]
        bag_feat = torch.bmm(attention, med_feat).squeeze(1)    # [B, D_inner]
        logits = self.classifier(bag_feat)                    # [B, n_class]
        return (logits, attention) if return_attention else logits


### Sanity check: does the model behave like MIL?

Before training, verify tensor shapes, attention normalization, **permutation invariance**, and **padding invariance**. Reordering a bag should not change its prediction; adding masked padding should not change it either.


In [ ]:
model_cpu = ABMIL(conf).eval()
x_demo = torch.randn(2, 7, conf.D_feat)
mask_demo = torch.ones(2, 7, dtype=torch.bool)
mask_demo[1, 4:] = False

with torch.no_grad():
    logits_demo, attn_demo = model_cpu(x_demo, return_attention=True, mask=mask_demo)
    unpadded_logits = model_cpu(x_demo[1:2, :4])
    permutation = torch.randperm(7)
    permuted_logits = model_cpu(x_demo[:, permutation], mask=mask_demo[:, permutation])

assert logits_demo.shape == (2, int(conf.n_class))
assert attn_demo.shape == (2, 1, 7)
torch.testing.assert_close(attn_demo.sum(dim=-1), torch.ones(2, 1))
torch.testing.assert_close(attn_demo[1, 0, 4:], torch.zeros(3))
torch.testing.assert_close(logits_demo[1:2], unpadded_logits)
torch.testing.assert_close(logits_demo, permuted_logits)

print('logits:', logits_demo.shape, '| attention:', attn_demo.shape)
print('Shape, normalization, padding, and permutation checks passed.')
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
ax.bar(np.arange(7) - 0.2, attn_demo[0, 0].numpy(), width=0.4, label='Slide A: 7 patches')
ax.bar(np.arange(7) + 0.2, attn_demo[1, 0].numpy(), width=0.4, label='Slide B: 4 patches + padding')
ax.set(title='Untrained attention: padding receives zero weight',
       xlabel='Patch index', ylabel='Attention weight')
ax.legend()
show_and_save(fig, 'attention_sanity_check.png')
del model_cpu


**Questions:**

- Which tensors in your model contain trainable parameters?
- What properties do attention weights have after softmax?
- Could we replace softmax with another function?


# Step 3 — Device, loss, and optimizer

Cross-entropy compares **raw class logits** against integer slide labels. Do not apply softmax before passing logits to `CrossEntropyLoss`.

AdamW updates all trainable projection, attention, and classifier parameters. The image feature extractor is already pretrained and is not trained here.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Reset after exploratory inspections for reproducible model initialization.
seed_everything(int(conf.seed))
model = ABMIL(conf).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=float(conf.lr), weight_decay=float(conf.wd)
)
print('Device:', device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


# Step 4 — One training epoch

Watch the sequence in the loop:

`train()` → move batch to device → `zero_grad()` → model → loss → `backward()` → `step()`.

Only this function updates the model. The mask travels with the input so attention never assigns weight to padded patches.


In [ ]:
def train_one_epoch(model, criterion, data_loader, optimizer, device, progress=True):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    batches = tqdm(data_loader, desc='Training slides', leave=False, disable=not progress)
    for data in batches:
        bag = data['input'].to(device, dtype=torch.float32)
        labels = data['label'].to(device, dtype=torch.long)
        mask = data['mask'].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(bag, mask=mask)
        loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite training loss; check input features.')
        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    if total_examples == 0:
        raise ValueError('The training DataLoader is empty.')
    return {
        'loss': total_loss / total_examples,
        'acc': 100.0 * total_correct / total_examples,
    }


# Step 5 - Validation and test evaluation

Evaluation should not update the model. Complete the missing lines and use the provided metric helper for macro-AUROC and macro-F1.


In [ ]:
@torch.no_grad()
def evaluate(model, criterion, data_loader, device, conf, return_predictions=False):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    probabilities_all, labels_all, slide_ids = [], [], []

    for data in data_loader:
        bag = data['input'].to(device, dtype=torch.float32)
        labels = data['label'].to(device, dtype=torch.long)
        mask = data['mask'].to(device)
        logits = model(bag, mask=mask)
        loss = criterion(logits, labels)
        probabilities = F.softmax(logits, dim=-1)

        total_loss += loss.item() * labels.shape[0]
        total_examples += labels.shape[0]
        # Keep only small prediction tensors, not slide features, across batches.
        probabilities_all.append(probabilities.cpu())
        labels_all.append(labels.cpu())
        slide_ids.extend(data['slide_id'])

    if total_examples == 0:
        raise ValueError('The evaluation DataLoader is empty.')
    probabilities_all = torch.cat(probabilities_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)
    metrics = classification_metrics(probabilities_all, labels_all, conf.n_class)
    metrics['loss'] = total_loss / total_examples
    if return_predictions:
        return metrics, probabilities_all, labels_all, slide_ids
    return metrics


**Checkpoint questions:**

- Why are `model.eval()` and `torch.no_grad()` different concepts?
- What would happen if gradients were accumulated because `optimizer.zero_grad()` was omitted?
- Why must the test set not be used to choose the best epoch?


# Step 6 — Train, validate, and select the best checkpoint

The test set remains untouched during training. Choose the checkpoint with the largest **validation macro-AUROC + macro-F1**, exactly as in the starter notebook.

The first run uses **two epochs and all training slides**. The progress bar counts slides (with the default batch size of one). A short run proves the pipeline works; it is not a converged model or a completed baseline.

The YAML contains `warmup_epoch` and `n_res`, but the starter ABMIL loop does not use them. This implementation does not silently add a learning-rate schedule or residual blocks.


In [ ]:
best_score = float('-inf')
best_epoch = -1
best_model_state = None
history = []

for epoch in range(NUM_EPOCHS):
    start = time.time()
    train_metrics = train_one_epoch(model, criterion, train_loader, optimizer, device)
    val_metrics = evaluate(model, criterion, val_loader, device, conf)
    score = val_metrics['auc'] + val_metrics['f1']

    if score > best_score:
        best_score = score
        best_epoch = epoch
        best_model_state = {
            name: value.detach().cpu().clone() for name, value in model.state_dict().items()
        }
        torch.save({
            'model_state_dict': best_model_state,
            'config': run_config,
            'epoch': epoch + 1,
            'validation_score': best_score,
        }, RUN_DIR / 'best_model.pt')

    row = {
        'epoch': epoch + 1,
        'train_loss': train_metrics['loss'], 'train_acc': train_metrics['acc'],
        'val_loss': val_metrics['loss'], 'val_acc': val_metrics['acc'],
        'val_auc': val_metrics['auc'], 'val_f1': val_metrics['f1'],
        'seconds': time.time() - start,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"train loss={row['train_loss']:.4f} acc={row['train_acc']:.2f}% | "
          f"val loss={row['val_loss']:.4f} auc={row['val_auc']:.3f} f1={row['val_f1']:.3f} | "
          f"best={best_epoch+1} | {row['seconds']:.1f}s", flush=True)

assert best_model_state is not None
print(f'Best validation epoch: {best_epoch + 1}; checkpoint saved in {RUN_DIR.name}')


In [ ]:
history_frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout='constrained')
axes[0].plot(history_frame.epoch, history_frame.train_loss, 'o-', label='Train')
axes[0].plot(history_frame.epoch, history_frame.val_loss, 'o-', label='Validation')
axes[0].set(title='Is the model learning?', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[1].plot(history_frame.epoch, history_frame.train_acc, 'o-', label='Train')
axes[1].plot(history_frame.epoch, history_frame.val_acc, 'o-', label='Validation')
axes[1].set(title='Slide classification accuracy', xlabel='Epoch', ylabel='Accuracy (%)', ylim=(0, 100))
axes[2].plot(history_frame.epoch, history_frame.val_auc, 'o-', label='Macro-AUROC')
axes[2].plot(history_frame.epoch, history_frame.val_f1, 'o-', label='Macro-F1')
axes[2].set(title='Validation checkpoint metrics', xlabel='Epoch', ylabel='Score', ylim=(0, 1))
for ax in axes:
    ax.axvline(best_epoch + 1, color='gray', linestyle=':', label='Selected epoch')
    ax.set_xticks(history_frame.epoch)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
show_and_save(fig, 'learning_curves.png')


# Step 7 - Final held-out test evaluation

Only now, after the best checkpoint has been chosen from validation performance, load it and evaluate once on the test set.


In [ ]:
model.load_state_dict(best_model_state)
test_metrics, test_probabilities, test_labels, test_slide_ids = evaluate(
    model, criterion, test_loader, device, conf, return_predictions=True
)
print('Selected validation epoch:', best_epoch + 1)
print(f'{RUN_MODE.upper()} held-out test results:')
for key, value in test_metrics.items():
    print(f'  {key}: {value:.4f}' + ('%' if key == 'acc' else ''))
if RUN_MODE == 'explore':
    print('These are short exploration results, not the final baseline.')

predicted_labels = test_probabilities.argmax(dim=1)
predictions = pd.DataFrame({
    'slide_id': test_slide_ids, 'label': test_labels.numpy(),
    'prediction': predicted_labels.numpy(),
})
for index in range(int(conf.n_class)):
    predictions[f'p_class_{index}'] = test_probabilities[:, index].numpy()
predictions.to_csv(RUN_DIR / 'test_predictions.csv', index=False)
(RUN_DIR / 'test_metrics.json').write_text(json.dumps({
    'run_mode': RUN_MODE, 'epochs': NUM_EPOCHS,
    'best_epoch': best_epoch + 1, **test_metrics,
}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
matrix = confusion_matrix(test_labels.numpy(), predicted_labels.numpy(),
                          labels=list(range(int(conf.n_class))))
ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap='Blues', colorbar=False,
)
axes[0].set_title(f'{DATASET}: where does the model make mistakes?')
correct = predicted_labels == test_labels
confidence = test_probabilities.max(dim=1).values
axes[1].hist([confidence[correct].numpy(), confidence[~correct].numpy()],
             bins=np.linspace(0, 1, 11), label=['Correct', 'Incorrect'], stacked=True)
axes[1].set(title='Confidence is not a guarantee of correctness',
            xlabel='Largest predicted class probability', ylabel='Test slides')
axes[1].legend()
show_and_save(fig, 'test_predictions.png')


# Step 8 — See which patches receive attention

The left plot places patches at their recorded slide coordinates. It does **not** reconstruct tissue pixels. The middle plot shows how attention is distributed across patches, and the right plot shows the slide-level prediction.

Attention weights sum to one **within each slide**. A high weight means “important to this learned representation,” not “this patch has that probability of cancer.” Do not compare raw attention values between slides with different numbers of patches as though they were probabilities of disease.

These are post-test diagnostics. Do not use them to tune the model against the held-out test set.


In [ ]:
one_batch = mil_collate_fn([test_data[0]])
model.eval()
with torch.no_grad():
    logits, attention = model(
        one_batch['input'].to(device, dtype=torch.float32),
        return_attention=True, mask=one_batch['mask'].to(device),
    )
weights = attention[0, 0].cpu().numpy()
probabilities = logits.softmax(dim=-1)[0].cpu().numpy()
coordinates = one_batch['coords'][0]
slide_id = one_batch['slide_id'][0]
print('Slide:', slide_id, '| true class:', int(one_batch['label'][0]),
      '| predicted class:', int(probabilities.argmax()))
print('Patches:', len(weights), '| attention sum:', float(weights.sum()))

fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout='constrained')
if coordinates is not None:
    xy = coordinates.numpy()
    points = axes[0].scatter(xy[:, 0], xy[:, 1], c=weights,
                             s=5, cmap='viridis', rasterized=True)
    axes[0].invert_yaxis()
    axes[0].set_aspect('equal', adjustable='datalim')
    axes[0].set(title='Spatial attention (coordinates, not tissue pixels)',
                xlabel='Slide x', ylabel='Slide y')
    fig.colorbar(points, ax=axes[0], label='Attention weight')
else:
    axes[0].text(0.5, 0.5, 'No coordinates stored for this slide.',
                 ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_axis_off()
sorted_weights = np.sort(weights)[::-1]
axes[1].plot(np.arange(1, len(weights) + 1), np.cumsum(sorted_weights))
axes[1].set(title='Does attention focus on a few patches?',
            xlabel='Patches, sorted by decreasing attention',
            ylabel='Cumulative attention', ylim=(0, 1.02))
axes[2].bar(CLASS_NAMES, probabilities)
axes[2].set(title='Slide-level class probabilities', ylabel='Probability', ylim=(0, 1))
axes[2].tick_params(axis='x', rotation=30)
show_and_save(fig, 'slide_attention.png')
np.savez_compressed(
    RUN_DIR / 'slide_attention.npz', attention=weights, probabilities=probabilities,
    slide_id=slide_id,
    **({'coords': coordinates.numpy()} if coordinates is not None else {}),
)


## Step 8b — A gallery of several WSIs

Each panel below is **one slide**. Its colored points are patch locations, not different patients or original tissue pixels.

The default shows six slides, balanced across available true classes in dataset order. Selection does **not** depend on confidence, correctness, or attention. In C16 this normally gives three normal and three tumor slides. This is an illustrative subset, not a replacement for the full test metrics.

- Change **`GALLERY_COUNT`** to show more or fewer slides.
- Set **`GALLERY_INDICES = [0, 10, 25, 40, 70, 100]`** to choose specific zero-based test indices.
- After Step 7, rerun **only this cell** to change the gallery. It reuses the trained model without training or updating it.
- Colors show **attention percentiles within each slide**. Bright yellow means relatively high attention in that slide, not high tumor probability. Even weak differences receive ranks, so use the single-slide raw-weight plot to examine absolute attention concentration.
- Confidence is the **slide-level predicted-class probability**, not confidence that every highlighted patch is diseased.
- Each panel preserves its aspect ratio but fits its own coordinate bounds. Sizes on screen are not a common physical scale.

This is post-test inspection. Do not use these test slides to choose training settings. No `.tif` images are required.


In [ ]:
from wsi_visualization import plot_attention_gallery, select_gallery_indices

GALLERY_COUNT = 6
GALLERY_INDICES = None  # Or choose explicit indices, e.g. [0, 10, 25, 40, 70, 100].

if best_model_state is None:
    raise RuntimeError('Run Step 7 first to load the selected trained model.')
if list(test_data.data_names) != list(test_slide_ids):
    raise ValueError('Test predictions and dataset order differ; rerun Step 7.')

gallery_indices = (
    select_gallery_indices(test_labels.cpu().numpy(), GALLERY_COUNT)
    if GALLERY_INDICES is None else list(GALLERY_INDICES)
)
# Restore the validation-selected model, even if another cell used model afterward.
model.load_state_dict(best_model_state)
gallery_figure, gallery_rows = plot_attention_gallery(
    model, test_data, gallery_indices, CLASS_NAMES,
)
gallery_summary = pd.DataFrame(gallery_rows)
gallery_summary.to_csv(RUN_DIR / 'attention_gallery.csv', index=False)
show_and_save(gallery_figure, 'attention_gallery.png')
print(gallery_summary[
    ['test_index', 'slide_id', 'true_class', 'predicted_class', 'confidence', 'correct', 'patches']
].to_string(index=False))
print('\nSaved gallery:', RUN_DIR / 'attention_gallery.png')


## Visualize Attention on Whole-Slide Images (WSIs)

Before running the attention visualization, please complete the following steps:

1. Visit the [ECE1512-WSI-Feature dataset on Hugging Face](https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature)

2. Download the original WSIs `test_067.tif` and `test_068.tif` and the corresponding tumor annotation file `test_068.xml`.

3. Create a folder named `slide/` under your project directory and place the downloaded files in this folder.

Run this cell after training and final test evaluation, with `DATASET = 'C16'`. It restores `best_model_state` and plots the two slides in `PROJECT_DIR/slide`:

- **test_068 (tumor):** attention overlay and green contours from `test_068.xml`, including a zoom of the annotated region.
- **test_067 (normal):** attention overlay without annotation contours, including a zoom around the highest-attention patch.

The blue rectangle marks the zoom shown on the right. Colors show **attention percentiles within each slide**, not patch tumor probabilities; areas without extracted patches stay transparent. The displayed `P(tumor)` is the model's slide-level prediction.

HDF5 patch coordinates and XML contour coordinates are assumed to be expressed in level-0 `(x, y)`  coordinates; patch coordinates are treated as top-left corners. The function parameter `patch_size_level0=512` is an editable **assumption** for the patch footprint because these HDF5 files do not store patch-size metadata. Match it to the feature extractor's patch size for an exact footprint. All visualization functions are in `wsi_visualization.py`; the cell below only calls `visualize_wsi_attention`.

[OpenSlide Python](https://openslide.org/api/python/) reads a thumbnail and a pyramid-level crop rather than loading the full-resolution WSI into memory. If dependencies are missing, run `%pip install openslide-python openslide-bin` and restart the kernel if needed. Raw per-patch attention and coordinates are retained in `attention_visualizations`.


In [ ]:
attention_visualizations = {}
if DATASET == 'C16' and SHOW_WSI:
    required_slides = ['test_067.tif', 'test_068.tif', 'test_068.xml']
    missing_slides = [name for name in required_slides
                      if not (PROJECT_DIR / 'slide' / name).is_file()]
    if missing_slides:
        print('Optional tissue-image overlays skipped. Missing:', ', '.join(missing_slides))
        print('The spatial attention plot above works from the feature coordinates.')
        print('For tissue overlays, add the original files to:', PROJECT_DIR / 'slide')
        print('Source: https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature')
    else:
        try:
            import openslide
        except (ImportError, OSError):
            print('Optional WSI reader unavailable. Install openslide-python and openslide-bin,')
            print('restart the kernel, and run the notebook again.')
        else:
            attention_visualizations = visualize_wsi_attention(
                model, conf, PROJECT_DIR, best_model_state
            )
else:
    print('Tissue-image overlays are optional and are configured only for C16.')
print('\nRun complete. Saved outputs:', RUN_DIR)


# Research extensions

Both proposals are implemented and evaluated: (1) detached full-to-masked prediction consistency with 90% patch retention (`masked_view_training.py`), and (2) gated eight-neighbor spatial context (`spatial_context.py`). `reproduce.py` runs their matched comparison. Motivation, equations, results, negative findings and limitations are in Sections V–XI of `report/report.md`.
